# Stage 05: Data Storage

This notebook creates a reproducible storage workflow for the S&P 500 volatility project. It preserves a raw CSV snapshot, writes a processed Parquet table, validates both reloads, and saves JSON data lineage.

## Storage Decisions

- **Raw CSV:** preserves the incoming Stage 04 API fields in a portable, inspectable format.
- **Processed Parquet:** stores typed analysis data and derived returns efficiently.
- **JSON manifest:** preserves nested provenance metadata that would not fit naturally in a table.
- **Environment-driven paths:** keep the notebook portable between machines without hard-coded user directories.

In [1]:
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / 'homework' / 'homework05').exists()
)
HOMEWORK_DIR = REPO_ROOT / 'homework' / 'homework05'
load_dotenv(HOMEWORK_DIR / '.env')
RAW_DIR = HOMEWORK_DIR / os.environ.get('DATA_DIR_RAW', 'data/raw')
PROCESSED_DIR = HOMEWORK_DIR / os.environ.get('DATA_DIR_PROCESSED', 'data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(HOMEWORK_DIR / 'src'))
from storage_utils import read_dataframe, validate_round_trip, write_dataframe

print('Raw directory:', RAW_DIR.resolve())
print('Processed directory:', PROCESSED_DIR.resolve())

Raw directory: /Users/yifang/bootcamp_Yifang_Qiu/homework/homework05/data/raw
Processed directory: /Users/yifang/bootcamp_Yifang_Qiu/homework/homework05/data/processed


## Load the Stage 04 Source Snapshot
The most recent versioned Yahoo Finance CSV from Stage 04 is the input for this storage exercise.

In [2]:
stage04_raw_dir = REPO_ROOT / 'homework' / 'homework04' / 'data' / 'raw'
source_files = sorted(stage04_raw_dir.glob('api_yahoo-finance_GSPC_*.csv'))
if not source_files:
    raise FileNotFoundError('Run the Stage 04 notebook first so an API CSV is available.')

source_path = source_files[-1]
source_data = pd.read_csv(source_path, parse_dates=['date'])
required_columns = ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']
if source_data.empty or not set(required_columns).issubset(source_data.columns):
    raise ValueError('Stage 04 source data is empty or has an unexpected schema.')

source_data.info()
source_data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    20 non-null     datetime64[ns]
 1   ticker  20 non-null     object        
 2   open    20 non-null     float64       
 3   high    20 non-null     float64       
 4   low     20 non-null     float64       
 5   close   20 non-null     float64       
 6   volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 1.2+ KB


,date,ticker,open,high,low,close,volume
0,2025-01-02 14:30:00,^GSPC,5903.259766,5935.089844,5829.529785,5868.549805,3621680000
1,2025-01-03 14:30:00,^GSPC,5891.069824,5949.339844,5888.660156,5942.470215,3667340000
2,2025-01-06 14:30:00,^GSPC,5982.810059,6021.040039,5960.009766,5975.379883,4940120000
3,2025-01-07 14:30:00,^GSPC,5993.259766,6000.680176,5890.680176,5909.029785,4517330000
4,2025-01-08 14:30:00,^GSPC,5910.660156,5927.890137,5874.779785,5918.250000,4441740000


## Save Raw CSV and Processed Parquet
The raw CSV keeps the input fields unchanged. The processed table adds `daily_return`, which belongs in the processed layer rather than the raw layer.

In [3]:
run_timestamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
raw_output_path = RAW_DIR / f'sp500_prices_{run_timestamp}.csv'
processed_output_path = PROCESSED_DIR / f'sp500_prices_with_returns_{run_timestamp}.parquet'

processed_data = source_data.sort_values('date').copy()
processed_data['daily_return'] = processed_data['close'].pct_change()

write_dataframe(source_data, raw_output_path)
write_dataframe(processed_data, processed_output_path)
print('Saved raw CSV:', raw_output_path.name)
print('Saved processed Parquet:', processed_output_path.name)

Saved raw CSV: sp500_prices_20260820-045835.csv
Saved processed Parquet: sp500_prices_with_returns_20260820-045835.parquet


## Reload and Validate
A successful write is not enough. Reload both formats and check shape, column order, dates, and numeric prices.

In [4]:
reloaded_raw = read_dataframe(raw_output_path)
reloaded_processed = read_dataframe(processed_output_path)

raw_validation = validate_round_trip(source_data, reloaded_raw)
processed_validation = validate_round_trip(processed_data, reloaded_processed)
{'raw_csv': raw_validation, 'processed_parquet': processed_validation}

/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l1dcachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l2cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.l3cachesize'. Detail: [errno 1] Operation not permitted
/Users/runner/work/crossbow/crossbow/arrow/cpp/src/arrow/util/cpu_info.cc:239: IOError: sysctlbyname failed for 'hw.optional.neon'. Detail: [errno 1] Operation not permitted


{'raw_csv': {'shape_equal': True,
  'columns_equal': True,
  'date_is_datetime': True,
  'close_is_numeric': True},
 'processed_parquet': {'shape_equal': True,
  'columns_equal': True,
  'date_is_datetime': True,
  'close_is_numeric': True}}

## Save JSON Data Lineage
JSON is appropriate for this nested provenance record: it includes source details, validation results, and output paths without flattening nested values into table columns.

In [5]:
manifest = {
    'dataset': 'S&P 500 daily prices',
    'retrieved_for_storage_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': {
        'stage': 'Stage 04',
        'file': str(source_path.relative_to(REPO_ROOT)),
        'provider': 'Yahoo Finance public chart endpoint',
    },
    'schema': {column: str(dtype) for column, dtype in source_data.dtypes.items()},
    'outputs': {
        'raw_csv': str(raw_output_path.relative_to(REPO_ROOT)),
        'processed_parquet': str(processed_output_path.relative_to(REPO_ROOT)),
    },
    'validations': {'raw_csv': raw_validation, 'processed_parquet': processed_validation},
}
manifest_path = RAW_DIR / f'sp500_prices_manifest_{run_timestamp}.json'
with manifest_path.open('w', encoding='utf-8') as file:
    json.dump(manifest, file, indent=2)

with manifest_path.open(encoding='utf-8') as file:
    reloaded_manifest = json.load(file)
assert reloaded_manifest['outputs'] == manifest['outputs']
print('Saved and reloaded JSON manifest:', manifest_path.name)
reloaded_manifest

Saved and reloaded JSON manifest: sp500_prices_manifest_20260820-045835.json


{'dataset': 'S&P 500 daily prices',
 'retrieved_for_storage_at_utc': '2026-08-20T04:58:37.197094+00:00',
 'source': {'stage': 'Stage 04',
  'file': 'homework/homework04/data/raw/api_yahoo-finance_GSPC_20260820-0454.csv',
  'provider': 'Yahoo Finance public chart endpoint'},
 'schema': {'date': 'datetime64[ns]',
  'ticker': 'object',
  'open': 'float64',
  'high': 'float64',
  'low': 'float64',
  'close': 'float64',
  'volume': 'int64'},
 'outputs': {'raw_csv': 'homework/homework05/data/raw/sp500_prices_20260820-045835.csv',
  'processed_parquet': 'homework/homework05/data/processed/sp500_prices_with_returns_20260820-045835.parquet'},
 'validations': {'raw_csv': {'shape_equal': True,
   'columns_equal': True,
   'date_is_datetime': True,
   'close_is_numeric': True},
  'processed_parquet': {'shape_equal': True,
   'columns_equal': True,
   'date_is_datetime': True,
   'close_is_numeric': True}}}

## Assumptions and Risks

- Stage 04's versioned raw CSV is treated as the source of truth for this exercise.
- CSV is intentionally retained for raw transparency, even though it does not preserve all types as precisely as Parquet.
- Parquet requires `pyarrow`; it is listed in `project/requirements.txt`.
- File names are retrieval snapshots, so later runs create new versions rather than silently overwriting earlier data.
- Date alignment, corporate actions, and market-data licensing remain important considerations for later modeling stages.